# Estudo de Correlação: Produto A vs Produto B
## Existe relação entre as vendas de dois produtos complementares?

**Contexto do estudo:**  
Este estudo investiga se as vendas mensais de um **Produto A** estão correlacionadas com as vendas de um **Produto B**.
A hipótese é que clientes que compram um produto tendem a comprar o outro em meses seguintes, o que permitiria **usar as vendas de um para prever as do outro**.

**Técnicas utilizadas:**
- Correlação de Pearson
- Regressão Linear Simples
- Detecção de Outliers (IQR)
- Teste de Causalidade de Granger
- Modelo de Previsão (Média Móvel + Tendência)


## 1. Importação das bibliotecas

In [ ]:
import pandas as pd  # Manipulação e análise de dados em tabelas (DataFrames)
import numpy as np  # Operações matemáticas e geração de arrays numéricos
import matplotlib.pyplot as plt  # Criação de gráficos e visualizações
import matplotlib.ticker as mticker  # Formatação dos eixos dos gráficos (ex: mostrar valores em %)
from sklearn.linear_model import LinearRegression  # Modelo de regressão linear simples
from sklearn.metrics import r2_score  # Métrica R² para avaliar a qualidade do modelo
from scipy.stats import pearsonr  # Cálculo da Correlação de Pearson (coeficiente + p-valor)
from statsmodels.tsa.stattools import grangercausalitytests  # Teste de Causalidade de Granger (séries temporais)
import warnings  # Biblioteca de controle de avisos do Python
warnings.filterwarnings('ignore')  # Suprime avisos desnecessários para deixar o output mais limpo

print("Bibliotecas importadas com sucesso!")

## 2. Criação dos dados fictícios

> **O que é isso?**  
> Para esse exercício eu criei uma base de dados fictícia que **preserva os padrões estatísticos** do estudo original: tendência de crescimento, sazonalidade e uma correlação positiva entre os dois produtos.  Ela pode ser substituída pela base de dados que você deseja comparar e ver se a correlação é tão boa quanto nesse case.


In [ ]:
# =====================================================================
# Dados mensais de jan/2021 a jul/2025 — 31 meses
# Valores em toneladas (fictícios, baseados em padrões reais)
# =====================================================================

data = {
    "ANO_MES": [
        202101, 202102, 202103, 202104, 202105, 202106,
        202107, 202108, 202109, 202110, 202111, 202112,
        202201, 202202, 202203, 202204, 202205, 202206,
        202207, 202208, 202209, 202210, 202211, 202212,
        202301, 202302, 202303, 202304, 202305, 202306, 202307
    ],
    "TONS_PRODUTO_A": [
        31.7, 23.7, 31.0, 37.3, 39.3, 37.4,
        41.4, 43.7, 41.7, 56.1, 61.9, 38.6,
        44.3, 47.8, 43.5, 49.3, 59.9, 59.0,
        65.4, 54.0, 59.2, 81.2, 84.0, 81.7,
        82.7, 79.0, 98.8, 122.1, 161.0, 170.2, 145.8
    ],
    "TONS_PRODUTO_B": [
        739.7, 664.2, 862.3, 713.7, 821.0, 767.3,
        859.6, 929.1, 870.0, 936.3, 1345.1, 897.3,
        969.0, 994.1, 799.1, 1044.8, 1110.7, 922.5,
        1034.5, 974.8, 1087.9, 1271.1, 1398.4, 1201.3,
        1150.0, 1089.4, 1312.5, 1489.3, 1601.2, 1720.5, 1580.4
    ]
}

df = pd.DataFrame(data)

# Converter ANO_MES para formato de data legível
df['DATA'] = pd.to_datetime(df['ANO_MES'].astype(str), format='%Y%m')

print(f"Base criada com {len(df)} meses de dados")
print(f"Período: {df['DATA'].min().strftime('%b/%Y')} até {df['DATA'].max().strftime('%b/%Y')}")
print()
print(df[['DATA', 'TONS_PRODUTO_A', 'TONS_PRODUTO_B']].head(5).to_string(index=False))


## 3. Exploração inicial dos dados (EDA)

> **Por que fazer isso primeiro?**  
> Antes de calcular qualquer correlação, precisamos entender e validar a base de dados. Essa etapa responde perguntas essenciais:
> - Os dados estão completos? Há valores faltando?
> - Temos o período e o volume de registros que esperávamos?
> - Os valores fazem sentido — ou há algo claramente errado (como toneladas negativas)?
> - As colunas que precisamos para o estudo estão presentes?
>
> Só depois de responder essas perguntas podemos confiar que a análise não será distorcida por problemas nos dados.


In [ ]:
# df.info() mostra um resumo estrutural da base:
# - quantas linhas e colunas temos
# - o nome e tipo de cada coluna (int, float, object, datetime...)
# - quantos valores não-nulos existem em cada coluna (útil para detectar dados faltantes)
# É o primeiro passo para saber se a base está completa e no formato esperado

print("Informações gerais:")
df.info()


In [ ]:
# df.describe() gera estatísticas descritivas automáticas para colunas numéricas:
# - count: quantos valores não-nulos (confirma se há dados faltantes)
# - mean: média — o valor central esperado
# - std: desvio padrão — o quanto os valores variam em torno da média
# - min / max: valores extremos — ajuda a identificar possíveis outliers
# - 25%, 50%, 75%: quartis — mostram a distribuição dos dados
# O .round(2) apenas arredonda para 2 casas decimais, facilitando a leitura

print("Estatísticas descritivas:")
df[['TONS_PRODUTO_A', 'TONS_PRODUTO_B']].describe().round(2)


In [ ]:
# df.isnull() identifica célula por célula se o valor está vazio (nulo)
# O .sum() soma esses nulos por coluna — resultado 0 significa que está tudo preenchido
# Esse check é obrigatório antes de qualquer análise: dados nulos podem
# distorcer médias, correlações e quebrar modelos sem dar nenhum erro visível

print("Valores nulos por coluna:")
print(df.isnull().sum())


## 4. Evolução mensal das vendas

> **O que vamos ver aqui?**  
> Plotamos a série temporal dos dois produtos para ter uma **visão temporal da evolução dos dados** antes de qualquer cálculo.  Se as curvas sobem e descem juntas, isso já é um sinal visual de correlação.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Produto A ---
axes[0].plot(df['DATA'], df['TONS_PRODUTO_A'], color='#2C7BB6', marker='o', linewidth=2, label='Produto A')
axes[0].fill_between(df['DATA'], df['TONS_PRODUTO_A'], alpha=0.15, color='#2C7BB6')
axes[0].set_title('Produto A — Evolução Mensal (ton)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Toneladas')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].legend()

# --- Produto B ---
axes[1].plot(df['DATA'], df['TONS_PRODUTO_B'], color='#FB8234', marker='s', linewidth=2, label='Produto B')
axes[1].fill_between(df['DATA'], df['TONS_PRODUTO_B'], alpha=0.15, color='#FB8234')
axes[1].set_title('Produto B — Evolução Mensal (ton)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Toneladas')
axes[1].set_xlabel('Mês')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('01_evolucao_mensal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo!")


## 5. Detecção e remoção de Outliers

> **O que são outliers e por que removê-los?**  
> Outliers são valores extremos que fogem muito do padrão — por exemplo, um mês com volume 10x maior que o normal por causa de uma ação pontual, sazonalidade, erro de cadastro ou evento extraordinário.  
> Se não tratarmos, eles podem **distorcer toda a correlação**, fazendo parecer que há (ou não há) uma relação que na verdade não existe com os dados normais.
>
> **Método IQR (Intervalo Interquartil):**  
> Considera outlier qualquer ponto abaixo de `Q1 - 1.5*IQR` ou acima de `Q3 + 1.5*IQR`.


In [ ]:
# Criamos uma função para reaproveitar a mesma lógica nos dois produtos
# sem precisar repetir o código duas vezes
def detectar_outliers_iqr(series):
    """
    Detecta outliers usando o método IQR.
    Retorna uma máscara booleana: True onde há outlier.
    """
    # Q1 = valor abaixo do qual estão 25% dos dados (1º quartil)
    # Q3 = valor abaixo do qual estão 75% dos dados (3º quartil)
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    # IQR = distância entre Q1 e Q3 — representa a "faixa central" dos dados
    IQR = Q3 - Q1

    # Qualquer valor fora dessa faixa é considerado outlier
    # O fator 1.5 é a convenção estatística padrão (definida por John Tukey)
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    print(f"  Q1={Q1:.1f} | Q3={Q3:.1f} | IQR={IQR:.1f}")
    print(f"  Limites: [{limite_inferior:.1f}, {limite_superior:.1f}]")

    # Retorna True para cada linha que está fora dos limites
    # O operador | significa "ou" — basta estar abaixo OU acima para ser outlier
    return (series < limite_inferior) | (series > limite_superior)

print("Produto A:")
outlier_a = detectar_outliers_iqr(df['TONS_PRODUTO_A'])

print()
print("Produto B:")
outlier_b = detectar_outliers_iqr(df['TONS_PRODUTO_B'])

# Um mês é marcado como outlier se qualquer um dos produtos tiver valor extremo
# Isso garante que não comparamos meses "normais" com meses atípicos
df['is_outlier'] = outlier_a | outlier_b

print()
print(f"Total de outliers encontrados: {df['is_outlier'].sum()} meses")

# Exibe apenas as linhas marcadas como outlier para inspeção visual
print(df[df['is_outlier']][['DATA', 'TONS_PRODUTO_A', 'TONS_PRODUTO_B']])


In [ ]:
# O operador ~ inverte a máscara booleana — ou seja, seleciona apenas as linhas onde is_outlier é False
# .copy() cria uma cópia independente para não afetar o df original nas próximas etapas
df_limpo = df[~df['is_outlier']].copy()

print(f"Base original: {len(df)} meses")
print(f"Base sem outliers: {len(df_limpo)} meses")


## 6. Correlação de Pearson

> **O que é a Correlação de Pearson?**  
> É uma medida que vai de **-1 a +1** e indica o quanto duas variáveis se movem juntas:  
> - **+1** → quando uma sobe, a outra sempre sobe também (correlação perfeita positiva)  
> - **0** → sem relação linear  
> - **-1** → quando uma sobe, a outra sempre cai  
>
> O **valor-p** indica se o resultado é estatisticamente significativo.  
> Convenção: p < 0.05 = podemos confiar que a correlação não é por acaso.


In [ ]:
# --- Correlação com todos os dados ---
corr_total, pval_total = pearsonr(df['TONS_PRODUTO_A'], df['TONS_PRODUTO_B'])

# --- Correlação sem outliers ---
corr_limpo, pval_limpo = pearsonr(df_limpo['TONS_PRODUTO_A'], df_limpo['TONS_PRODUTO_B'])

print("Correlação de Pearson — Produto A vs Produto B")
print(f"\n  Com todos os dados:")
print(f"    r = {corr_total:.4f} | p-valor = {pval_total:.4e}")
print(f"\n  Sem outliers:")
print(f"    r = {corr_limpo:.4f} | p-valor = {pval_limpo:.4e}")

if corr_limpo > 0.7:
    print("\nCorrelação FORTE e positiva!")
elif corr_limpo > 0.4:
    print("\nCorrelação MODERADA.")
else:
    print("\nCorrelação FRACA.")


## 7. Regressão Linear Simples

> **O que é Regressão Linear?**  
> Ela nos diz: *"para cada X toneladas a mais do Produto A, quantas toneladas a mais do Produto B podemos esperar?"*  
>
> A fórmula é `y = a + b * x`, onde:  
> - **a** (intercepto) = valor base de B quando A é zero  
> - **b** (coeficiente) = inclinação da reta — o quanto B cresce para cada unidade adicional de A  
> - **R²** = quanto da variação de B é explicada por A (vai de 0 a 1; quanto mais próximo de 1, melhor o ajuste)  
>
> O gráfico de dispersão gerado nessa etapa é especialmente útil para apresentações a stakeholders:  
> ele traduz a correlação em algo visual e intuitivo, mostrando graficamente o quanto as duas variáveis  
> se movem juntas — sem precisar explicar nenhuma fórmula.


In [ ]:
# --- Regressão: Produto A → Produto B ---
X = df_limpo[['TONS_PRODUTO_A']].values
y = df_limpo['TONS_PRODUTO_B'].values

modelo = LinearRegression()
modelo.fit(X, y)
y_pred = modelo.predict(X)
r2 = r2_score(y, y_pred)

print("Regressão Linear: Produto A → Produto B")
print(f"   Intercepto (a): {modelo.intercept_:.2f}")
print(f"   Coeficiente (b): {modelo.coef_[0]:.2f}")
print(f"   R²: {r2:.4f}")
print()
print(f"   Interpretação: Para cada +1 tonelada de Produto A,")
print(f"   esperamos +{modelo.coef_[0]:.1f} toneladas de Produto B.")


In [ ]:
# --- Gráfico de dispersão com reta de regressão ---
fig, ax = plt.subplots(figsize=(12, 7))

# =====================================================================
# CORES — personalize aqui de acordo com a identidade visual do cliente
# Dica: use códigos HEX (#RRGGBB) ou nomes em inglês (ex: 'blue', 'red')
# =====================================================================
cor_pontos    = '#2C7BB6'     # azul — pontos dos meses normais
cor_outliers  = 'red'         # vermelho — pontos dos meses excluídos (outliers)
cor_reta      = 'darkgreen'   # verde escuro — linha de regressão
cor_anotacao  = 'navy'        # azul escuro — texto da anotação de correlação
cor_caixa     = 'lightyellow' # amarelo claro — fundo da caixa de anotação
# =====================================================================

# Pontos normais
ax.scatter(df_limpo['TONS_PRODUTO_A'], df_limpo['TONS_PRODUTO_B'],
           color=cor_pontos, s=80, alpha=0.85, label='Meses normais', zorder=3)

# Outliers em destaque
outliers = df[df['is_outlier']]
ax.scatter(outliers['TONS_PRODUTO_A'], outliers['TONS_PRODUTO_B'],
           color=cor_outliers, s=100, alpha=0.7, marker='X', label='Outliers (excluídos)', zorder=4)

# Reta de regressão
x_range = np.linspace(df_limpo['TONS_PRODUTO_A'].min(), df_limpo['TONS_PRODUTO_A'].max(), 100)
y_trend = modelo.predict(x_range.reshape(-1, 1))
ax.plot(x_range, y_trend, color=cor_reta, linestyle='--', linewidth=2,
        label=f'Regressão Linear (R²={r2:.2f})')

ax.set_title('Dispersão: Produto A vs Produto B', fontsize=14, fontweight='bold')
ax.set_xlabel('Toneladas — Produto A', fontsize=12)
ax.set_ylabel('Toneladas — Produto B', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)

# Anotação movida para o canto inferior direito — sem sobreposição com a legenda
ax.annotate(f'r = {corr_limpo:.3f}\np = {pval_limpo:.3e}',
            xy=(0.75, 0.08), xycoords='axes fraction',
            fontsize=12, color=cor_anotacao,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=cor_caixa, alpha=0.8))

plt.tight_layout()
plt.savefig('02_dispersao_regressao.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo!")


## 8. Teste de Causalidade de Granger

> **O que é o Teste de Granger?**  
> Correlação não é causalidade — duas coisas podem crescer juntas por coincidência ou por um terceiro fator em comum.  
> O teste de Granger responde uma pergunta mais precisa: *"o histórico do Produto A ajuda a **prever** o futuro do Produto B melhor do que o histórico de B sozinho?"*  
>
> Se sim, dizemos que A **Granger-causa** B — o que sugere uma relação de influência temporal entre as variáveis.  
>
> **Lag** = defasagem em meses, ou seja, de quanto tempo atrás estamos olhando:  
> - Lag 1 = o valor de A no mês passado prevê B neste mês  
> - Lag 2 = o valor de A há 2 meses prevê B neste mês  
> - Lag 3 = o valor de A há 3 meses prevê B neste mês  
>
> Testamos múltiplos lags porque a influência entre produtos pode não ser imediata —  
> um cliente que compra A em janeiro pode demorar alguns meses para comprar B.  
>
> **Como interpretar o resultado:**  
> - p-valor < 0.05 → há evidência estatística de que A influencia B naquele lag  
> - p-valor >= 0.05 → não há evidência suficiente para afirmar causalidade  
>
> No gráfico de barras, as barras verdes indicam lags significativos e as vermelhas, não significativos.


In [ ]:
# Preparar série temporal ordenada
# Usamos apenas as colunas necessárias e ordenamos por data para garantir a sequência temporal correta
df_granger = df_limpo[['DATA', 'TONS_PRODUTO_A', 'TONS_PRODUTO_B']].copy()
df_granger = df_granger.sort_values('DATA').set_index('DATA')

print("Teste de Causalidade de Granger: Produto A → Produto B")
print("=" * 55)

# A ordem das colunas importa: [variável a ser prevista, variável preditora]
resultados = grangercausalitytests(
    df_granger[['TONS_PRODUTO_B', 'TONS_PRODUTO_A']],
    maxlag=3,
    verbose=True
)

# Interpretação automática por lag
print("\nInterpretação dos resultados:")
print("-" * 55)
for lag in range(1, 4):
    p = resultados[lag][0]['ssr_ftest'][1]
    if p < 0.05:
        print(f"Lag {lag}: p = {p:.4f} → ha evidencia estatistica de que Produto A influencia Produto B")
    else:
        print(f"Lag {lag}: p = {p:.4f} → nao ha evidencia suficiente de influencia neste lag")


In [ ]:
# Definindo o número máximo de lags a testar
# Quanto maior o lag, mais meses "para trás" estamos olhando para prever B
max_lag = 5

# Rodando o teste novamente com verbose=False para extrair os resultados manualmente
res = grangercausalitytests(
    df_granger[['TONS_PRODUTO_B', 'TONS_PRODUTO_A']],
    maxlag=max_lag, verbose=False
)

# Extraindo o p-valor do teste F (ssr_ftest) para cada lag
# ssr_ftest é o teste mais comum para Granger — compara os erros do modelo com e sem a variável preditora
p_valores = {lag: res[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag + 1)}

# Transformando em DataFrame para facilitar a visualização e o plot
df_pval = pd.DataFrame(list(p_valores.items()), columns=['Lag', 'p_valor'])

print("P-valores por Lag (Produto A → Produto B):")
print(df_pval.to_string(index=False))

# =====================================================================
# CORES — personalize de acordo com a identidade visual do cliente
cor_significativo     = 'green'   # lag com evidência estatística (p < 0.05)
cor_nao_significativo = 'salmon'  # lag sem evidência suficiente
cor_linha_referencia  = 'red'     # linha de corte de significância
# =====================================================================

# Gráfico de barras — cada barra representa um lag
fig, ax = plt.subplots(figsize=(9, 5))

# Colorir as barras: verde se significativo, salmão se não
cores = [cor_significativo if p < 0.05 else cor_nao_significativo for p in df_pval['p_valor']]
ax.bar(df_pval['Lag'], df_pval['p_valor'], color=cores, alpha=0.8, edgecolor='black')

# Linha de referência em p=0.05 — o limiar de significância estatística
ax.axhline(0.05, color=cor_linha_referencia, linestyle='--', linewidth=1.5, label='Significancia (p=0.05)')

ax.set_title('Granger: P-valores por Lag (Produto A → Produto B)', fontsize=13, fontweight='bold')
ax.set_xlabel('Lag (meses)')
ax.set_ylabel('P-valor')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Exibir o valor exato de p em cima de cada barra
for i, row in df_pval.iterrows():
    ax.text(row['Lag'], row['p_valor'] + 0.005, f"{row['p_valor']:.3f}", ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('03_granger_pvalores.png', dpi=150, bbox_inches='tight')  # salva na mesma pasta do notebook
plt.show()
print("Grafico salvo!")


In [ ]:
# Extrair e visualizar os p-valores por lag
max_lag = 5
res = grangercausalitytests(
    df_granger[['TONS_PRODUTO_B', 'TONS_PRODUTO_A']],
    maxlag=max_lag, verbose=False
)

p_valores = {lag: res[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag + 1)}
df_pval = pd.DataFrame(list(p_valores.items()), columns=['Lag', 'p_valor'])

print("P-valores por Lag (Produto A → Produto B):")
print(df_pval.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(9, 5))
cores = ['green' if p < 0.05 else 'salmon' for p in df_pval['p_valor']]
ax.bar(df_pval['Lag'], df_pval['p_valor'], color=cores, alpha=0.8, edgecolor='black')
ax.axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significância (p=0.05)')
ax.set_title('Granger: P-valores por Lag (Produto A → Produto B)', fontsize=13, fontweight='bold')
ax.set_xlabel('Lag (meses)')
ax.set_ylabel('P-valor')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
for i, row in df_pval.iterrows():
    ax.text(row['Lag'], row['p_valor'] + 0.005, f"{row['p_valor']:.3f}", ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('03_granger_pvalores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo!")


## 9. A relação inversa é verdadeira?

> **Por que testar o inverso?**  
> A correlação é simétrica (A vs B = B vs A), mas a **causalidade não é**.  
> Pode ser que B preveja A, e não o contrário — ou que ambos se prevejam mutuamente.  
> Testar os dois sentidos é fundamental para não tirar a conclusão errada.  
>
> Na prática, isso tem impacto direto nas decisões de negócio:  
> - Se apenas A causa B → usamos A como sinal para antecipar demanda de B  
> - Se apenas B causa A → invertemos a lógica de previsão  
> - Se os dois se causam mutuamente → temos uma relação complementar, e ambos podem ser usados como sinais um do outro em estratégias de cross-sell e planejamento


In [ ]:
# Correlação: Produto B → Produto A
corr_inv, pval_inv = pearsonr(df_limpo['TONS_PRODUTO_B'], df_limpo['TONS_PRODUTO_A'])

# Regressão: Produto B → Produto A
X_inv = df_limpo[['TONS_PRODUTO_B']].values
y_inv = df_limpo['TONS_PRODUTO_A'].values
modelo_inv = LinearRegression().fit(X_inv, y_inv)
r2_inv = r2_score(y_inv, modelo_inv.predict(X_inv))

print("Correlação inversa: Produto B → Produto A")
print(f"   r = {corr_inv:.4f} | p-valor = {pval_inv:.4e} | R² = {r2_inv:.4f}")

# Granger: B → A
print()
print("Granger: Produto B → Produto A")
res_inv = grangercausalitytests(
    df_granger[['TONS_PRODUTO_A', 'TONS_PRODUTO_B']],
    maxlag=3, verbose=False
)

for lag in range(1, 4):
    p = res_inv[lag][0]['ssr_ftest'][1]
    sig = "Significativo" if p < 0.05 else "Nao significativo"
    print(f"   Lag {lag}: p = {p:.4f}  →  {sig}")


### Resultado: a relação inversa também é verdadeira!

Os resultados mostram que **Produto B também Granger-causa Produto A** nos lags 1 e 3, com p-valores abaixo de 0.05. Isso significa que o histórico de vendas de B ajuda a prever as vendas futuras de A — confirmando uma **relação de influência mútua** entre os dois produtos.

**O que isso significa na prática:**
- Nos lags 1 e 3, quando as vendas de B sobem, as de A tendem a subir nos meses seguintes
- O lag 2 não foi significativo, sugerindo que a influência não é linear mês a mês
- A causalidade é bidirecional: **Produto B passou a influenciar Produto A, e não mais apenas o contrário**

**Implicação estratégica:**
- Campanhas de retargeting podem ser acionadas após compras de B, mirando A nos 30 a 90 dias seguintes
- Modelos preditivos e planejamento logístico devem considerar o histórico de ambos os produtos como variáveis de entrada


## 10. Visualização final: Séries sobrepostas

> **Por que essa visualização?**  
> Após confirmar estatisticamente a correlação e a causalidade entre os produtos, esse gráfico serve como a **peça de comunicação final** da análise.  
>
> Ele sobrepõe as duas séries temporais em um mesmo eixo de tempo, usando eixos Y independentes para respeitar a diferença de escala entre os produtos.  
> Isso torna o movimento conjunto visível de forma intuitiva — mesmo para quem não entende de estatística.  
>
> É o gráfico ideal para apresentar em reuniões com times comerciais, de marketing ou liderança,  
> onde o objetivo é mostrar o insight de forma clara e convincente, sem precisar explicar p-valores ou regressões.


In [ ]:
# =====================================================================
# CORES — personalize de acordo com a identidade visual do cliente
color_a = '#2C7BB6'  # azul — Produto A (eixo esquerdo)
color_b = '#FB8234'  # laranja — Produto B (eixo direito)
# =====================================================================

# Criamos dois eixos Y independentes (ax1 e ax2) porque os produtos têm
# escalas muito diferentes — sem isso, um produto "esmaga" o outro no gráfico
fig, ax1 = plt.subplots(figsize=(14, 6))

# --- Produto A — eixo Y esquerdo ---
ax1.set_xlabel('Mês', fontsize=12)
ax1.set_ylabel('Toneladas — Produto A', color=color_a, fontsize=12)
ax1.plot(df['DATA'], df['TONS_PRODUTO_A'], color=color_a, marker='o',
         linewidth=2, label='Produto A')
# Colorir os números do eixo Y com a mesma cor do produto — facilita a leitura
ax1.tick_params(axis='y', labelcolor=color_a)

# --- Produto B — eixo Y direito ---
# twinx() cria um segundo eixo Y compartilhando o mesmo eixo X
ax2 = ax1.twinx()
ax2.set_ylabel('Toneladas — Produto B', color=color_b, fontsize=12)
ax2.plot(df['DATA'], df['TONS_PRODUTO_B'], color=color_b, marker='s',
         linewidth=2, linestyle='--', label='Produto B')
ax2.tick_params(axis='y', labelcolor=color_b)

# --- Título e legenda ---
ax1.set_title('Evolução Conjunta: Produto A e Produto B (jan/21 – jul/23)',
              fontsize=14, fontweight='bold')

# Como temos dois eixos, precisamos combinar as legendas manualmente
# para que ambos os produtos apareçam na mesma caixa de legenda
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

plt.xticks(rotation=45)
ax1.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('04_series_sobrepostas.png', dpi=150, bbox_inches='tight')  # salva na pasta do notebook
plt.show()
print("Grafico salvo!")


## 11. Conclusões

### O que encontramos:

1. **Correlação de Pearson forte e positiva** entre Produto A e Produto B  
   → Os dois crescem juntos ao longo do tempo.

2. **Regressão Linear** mostra que B responde de forma previsível a A  
   → É possível usar A como variável explicativa de B.

3. **Teste de Granger** indica que **B Granger-causa A** nos lags de 1–3 meses  
   → Quando as vendas de B sobem, as de A tendem a subir nos meses seguintes.

4. **A relação inversa também é verdadeira**  
   → Os produtos se influenciam mutuamente, sugerindo uma **dinâmica complementar**.

### Implicações práticas:

- **Campanhas de cross-sell** podem ser acionadas após compras de qualquer um dos produtos
- **Planejamento de estoque** deve considerar ambos como sinalizadores um do outro
- **Modelos preditivos** devem incluir o histórico dos dois produtos como features

---

### Quando aplicar esse tipo de estudo?

Esse tipo de análise é útil sempre que houver uma dúvida de negócio do tipo:  
*"será que o desempenho de X influencia o desempenho de Y?"*

Alguns exemplos práticos:

- **Vendas:** as vendas de um produto de entrada preveem as vendas de um produto premium?
- **Marketing:** o aumento de investimento em mídia paga influencia as buscas orgânicas?
- **Operações:** o volume de pedidos de um canal antecipa o volume de outro canal?
- **Financeiro:** a variação de um indicador econômico influencia a receita da empresa?
- **Produto:** o engajamento com uma feature do app antecipa a adoção de outra feature?

Sempre que houver duas variáveis medidas ao longo do tempo e uma suspeita de que uma pode influenciar a outra, esse estudo pode ser aplicado para confirmar ou refutar essa hipótese com evidência estatística.
